# ADSM (Arcade / Five-foot way) shadow simulation -- standalone workflow (final 13 classes, finalised 2026-06-13)
Based on the `00-SG_self_Taihan [Zenodo code].ipynb` template, with exactly the same configuration as the successful citywide run (98.9 min / 77 tiles):

1. **Preprocessing (once per set of base rasters)**: three vectors -> the full set of rasters, grid strictly aligned to the DEM
   - `building_remain.gpkg` (height field) -> **DSMremain** (= DEM + height) + **BREMAIN** (class 12 mask)
   - `arcade.gpkg` (bld_h / arc_h) -> **ADSM** (box top) + **ADSMB** (box base, thin-box protection)
   - `covered_linkway pednet.gpkg` -> **LDSMpednet** (uniform 3 m)
   - CDSM cleaning (nodata/negative -> 0) -> **CDSMclean**; generate all-zero **WALLS0/ASPECT0**
2. `thermal_comfort(... arcade_filename, arcade_base_filename, building_remain_filename, zero walls)` -> **13-class shadow category**
3. **Zero walls**: pure-shadow (only_shadow) mode has been verified to give bit-for-bit the same output as real walls, saving 7-8 hours; real walls are only needed for future Tmrt/UTCI runs
4. Result statistics (hourly 13 classes + shadow share) and previews

> **Final class semantics**: 0 sunlit | 1 bldg | 2 veg | 3 b+v | 4 ldsm | 5 b+ldsm | 6 v+ldsm | 7 b+v+ldsm |
> **8 sheltered five-foot-way walkway** (shaded by the box or a building) | 9 +veg | 10 +ldsm | 11 +veg+ldsm | **12 building footprint (not shadow)**
> Rules: (1) BREMAIN -> 12; (2) strip pixels shaded by a building mass (box/main building) -> 8-11; (3) other building-involved pixels (incl. box shadow cast onto the street) -> 1/3/5/7; (4) vegetation/linkway only -> 2/4/6; (5) unshaded -> 0
> Shadow_*.tif values are pure physical quantities; the category remapping does not alter them (verified bit-for-bit).

In [ ]:
# %pip uninstall -y solweig-gpu
# %pip install -e D:\GitHub\SOLWEIG-GPU

#### Environment setup (reused from the 00 template)

In [ ]:
import sys
import os
import importlib

# ===== QGIS path =====
qgis_path = r'C:\Program Files\QGIS 3.40.15\apps\qgis-ltr'

# Add the QGIS Python library paths
sys.path.append(os.path.join(qgis_path, 'python'))
sys.path.append(r'C:\Program Files\QGIS 3.40.15\apps\qgis-ltr\python\plugins')
sys.path.append(r'C:\Users\City Syntax Lab\AppData\Roaming\QGIS\QGIS3\profiles\default\python\plugins')

os.environ['PATH'] += f';{os.path.join(qgis_path, "bin")}'
os.environ['QT_QPA_PLATFORM_PLUGIN_PATH'] = os.path.join(qgis_path, 'qtplugins')

# ===== Local GitHub project root =====
local_repo = r'D:\GitHub\SOLWEIG-GPU'
# local_repo = r'D:\[backup]\SOLWEIG-GPU'

# Must be placed first
if sys.path[0] != local_repo:
    if local_repo in sys.path:
        sys.path.remove(local_repo)
    sys.path.insert(0, local_repo)

print("sys.path[0] =", sys.path[0])

# ===== Force use of the local debug code =====
# If the notebook has already imported solweig_gpu, clear the cache first
to_delete = [m for m in sys.modules if m == "solweig_gpu" or m.startswith("solweig_gpu.")]
for m in to_delete:
    del sys.modules[m]

# ===== Initialise QGIS =====
from qgis.core import *
QgsApplication.setPrefixPath(qgis_path, True)
qgs = QgsApplication([], False)
qgs.initQgis()

import warnings
warnings.filterwarnings("ignore")

import datetime
from osgeo import gdal
import numpy as np
from osgeo.gdalconst import *
from PyQt5.QtCore import QDate, QTime

import processing
from processing_umep.processing_umep_provider import ProcessingUMEPProvider
umep_provider = ProcessingUMEPProvider()
QgsApplication.processingRegistry().addProvider(umep_provider)

from processing.core.Processing import Processing
Processing.initialize()

import matplotlib.pylab as plt
import numpy
import osgeo.gdal as gdal

print("numpy:", numpy.__version__)
print("gdal:", gdal.__version__)

import glob
import pandas as pd
from multiprocessing import Pool
from datetime import datetime, timedelta
import geopandas as gpd
from geopy.distance import geodesic
from sklearn.neighbors import NearestNeighbors
import math

In [ ]:
# ===== Re-import the local solweig_gpu =====
import solweig_gpu
import torch
importlib.reload(solweig_gpu)
print("solweig_gpu path:", solweig_gpu.__file__)

print("GDAL OK")
print("SOLWEIG-GPU version:", getattr(solweig_gpu, "__version__", "No __version__"))
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())

from solweig_gpu import shadow
importlib.reload(shadow)

print("shadow path:", shadow.__file__)
print("CUDA available check:", torch.cuda.is_available())

#### Global configuration

In [ ]:
import os
city = 'SG'            # 'SG' / 'BO'
date = '2026-03-01'
n = 1                  # resolution (m)

# ===== Input root directory (base rasters + Forcing_data) =====
base_folder = r'D:\Claude\SVI_FFW\TIF'
raster_path = base_folder                      # base rasters sit directly in base_folder
wea_path    = os.path.join(base_folder, 'Forcing_data', 'S50_Clementi Road.txt')

# ===== Three input vectors (EPSG must match the rasters, SG=3414) =====
building_remain_gpkg = r'D:\Claude\SVI_FFW\Shp\SG\step2_building_remain_sg.gpkg'   # field height (above ground, m)
arcade_gpkg          = r'D:\Claude\SVI_FFW\Shp\SG\step2_arcade_sg.gpkg'            # fields bld_h / arc_h
linkway_gpkg         = r'D:\Claude\SVI_FFW\Shp\SG\covered_linkway_SG_island_tv_pednet_bridged.gpkg'  # burned at a uniform 3 m

# ===== File naming (follows the SUB_{city}_Polygon_{layer}_{n}m.tif convention) =====
name = lambda layer: f'SUB_{city}_Polygon_{layer}_{n}m.tif'
DEM_IN   = name('DEM');  CDSM_IN = name('CDSM'); LC_IN = name('LC')
# ---- Generated by preprocessing (outputs of the prep cell in this notebook) ----
DSM_REM  = name('DSMremain')    # building DSM = DEM + building_remain.height (the arcade strip is naturally ground level)
BREMAIN  = name('BREMAIN')      # 0/1 mask -> class 12
ADSM_TOP = name('ADSM')         # arcade box top = bld_h (above ground)
ADSM_BAS = name('ADSMB')        # arcade box base = arc_h (above ground, thin-box protection)
LDSM_PED = name('LDSMpednet')   # covered linkway @3 m
CDSM_CLN = name('CDSMclean')    # cleaned canopy (nodata/negative -> 0)
W0 = name('WALLS0'); A0 = name('ASPECT0')   # all-zero walls (verified bit-for-bit equivalent in only_shadow mode)
print('base:', base_folder)

#### Preprocessing: three vectors -> the full set of rasters [run once per set of base rasters; windowed, memory safe]
- `DSMremain` = DEM + building_remain.height (building extent/height pixel-consistent with the vectors; the arcade strip is not part of remain, so it is naturally ground level and no "carving" is needed);
- `BREMAIN` = remain 0/1 mask (class 12); `ADSM/ADSMB` = box top/base (low buildings protected to "at least 0.5 m thick");
- `LDSMpednet` = covered linkway at a uniform 3 m; `CDSMclean` = cleaned canopy (nodata/negative -> 0; the program uses it as height above ground);
- `WALLS0/ASPECT0` = all-zero walls (pure-shadow mode verified bit-for-bit equivalent to real walls);
- Pure osgeo/gdal: vector rasterization writes straight to disk (GDAL blocks internally); raster composition is windowed in 2048-row bands, so even 44k x 27k does not exhaust memory.

In [ ]:
import numpy as np
from osgeo import gdal, ogr
gdal.UseExceptions()

REF = os.path.join(raster_path, DEM_IN)        # grid reference = DEM
_r = gdal.Open(REF); GT = _r.GetGeoTransform(); PROJ = _r.GetProjection()
W, H = _r.RasterXSize, _r.RasterYSize; _r = None
CO = ['COMPRESS=LZW', 'TILED=YES', 'BIGTIFF=YES']
print(f'grid {W} x {H}')

def new_raster(path, dtype=gdal.GDT_Float32):
    ds = gdal.GetDriverByName('GTiff').Create(path, W, H, 1, dtype, options=CO)
    ds.SetGeoTransform(GT); ds.SetProjection(PROJ)
    ds.GetRasterBand(1).Fill(0)
    return ds

def rasterize(gpkg, out, attr=None, burn=None, dtype=gdal.GDT_Float32):
    """Vector rasterization (pixel-centre rule), written straight to disk, GDAL blocks internally. Use either attr or burn."""
    ds = new_raster(out, dtype)
    src = ogr.Open(gpkg)
    if attr is not None:
        gdal.RasterizeLayer(ds, [1], src.GetLayer(0), options=[f'ATTRIBUTE={attr}'])
    else:
        gdal.RasterizeLayer(ds, [1], src.GetLayer(0), burn_values=[burn])
    src = None; ds.FlushCache(); ds = None
    print('rasterized ->', os.path.basename(out))

# ---------- (1) Vector rasterization ----------
TMP_H    = os.path.join(raster_path, '_tmp_height.tif')
TMP_BASE = os.path.join(raster_path, '_tmp_base.tif')
rasterize(building_remain_gpkg, TMP_H, attr='height')
rasterize(building_remain_gpkg, os.path.join(raster_path, BREMAIN), burn=1, dtype=gdal.GDT_Byte)
rasterize(arcade_gpkg, os.path.join(raster_path, ADSM_TOP), attr='bld_h')
rasterize(arcade_gpkg, TMP_BASE, attr='arc_h')
rasterize(linkway_gpkg, os.path.join(raster_path, LDSM_PED), burn=3.0)

# ---------- (2) Windowed composition ----------
BS = 2048
d_dem  = gdal.Open(REF)
d_h    = gdal.Open(TMP_H)
d_top  = gdal.Open(os.path.join(raster_path, ADSM_TOP))
d_bas  = gdal.Open(TMP_BASE, gdal.GA_Update)
d_cdsm = gdal.Open(os.path.join(raster_path, CDSM_IN))
o_rem  = new_raster(os.path.join(raster_path, DSM_REM))
o_cln  = new_raster(os.path.join(raster_path, CDSM_CLN))
n_b = n_strip = n_ovl = n_canopy = 0
for y0 in range(0, H, BS):
    hh = min(BS, H - y0)
    dem = d_dem.GetRasterBand(1).ReadAsArray(0, y0, W, hh).astype(np.float32)
    dem = np.where(np.isfinite(dem) & (dem > -100), dem, 0)
    hgt = d_h.GetRasterBand(1).ReadAsArray(0, y0, W, hh).astype(np.float32)
    top = d_top.GetRasterBand(1).ReadAsArray(0, y0, W, hh).astype(np.float32)
    bas = d_bas.GetRasterBand(1).ReadAsArray(0, y0, W, hh).astype(np.float32)
    # Thin-box protection: base = min(base, max(top-0.5, 0.5)); 0 where there is no strip (written back to TMP_BASE -> becomes ADSMB)
    bas = np.where(top > 0, np.minimum(bas, np.maximum(top - 0.5, 0.5)), 0).astype(np.float32)
    d_bas.GetRasterBand(1).WriteArray(bas, 0, y0)
    # DSMremain = DEM + height; overlap with the strip (theoretically 0) is safely reset to ground level
    ovl = (hgt > 0) & (top > 0)
    rem = dem + hgt; rem[ovl] = dem[ovl]
    o_rem.GetRasterBand(1).WriteArray(rem, 0, y0)
    # CDSMclean
    c = d_cdsm.GetRasterBand(1).ReadAsArray(0, y0, W, hh).astype(np.float32)
    c = np.where(np.isfinite(c) & (c > 0), c, 0)
    o_cln.GetRasterBand(1).WriteArray(c, 0, y0)
    n_b += int((hgt > 0).sum()); n_strip += int((top > 0).sum())
    n_ovl += int(ovl.sum());     n_canopy += int((c > 0).sum())
# Note: each dataset must be explicitly set to None to really close it (d=None inside a for loop does not close it!)
d_dem = None; d_h = None; d_top = None; d_cdsm = None
d_bas.FlushCache(); d_bas = None
o_rem.FlushCache(); o_rem = None; o_cln.FlushCache(); o_cln = None
# TMP_BASE has become ADSMB in place -> rename
adsmb_path = os.path.join(raster_path, ADSM_BAS)
if os.path.exists(adsmb_path): os.remove(adsmb_path)
os.rename(TMP_BASE, adsmb_path); os.remove(TMP_H)

# ---------- (3) All-zero walls ----------
for p in (W0, A0):
    z = new_raster(os.path.join(raster_path, p)); z.FlushCache(); z = None

print(f'building pixels {n_b:,} (should = BREMAIN mask count) | strip {n_strip:,} | strip overlap {n_ovl:,} (should = 0) | canopy {n_canopy:,}')
print('outputs:', DSM_REM, BREMAIN, ADSM_TOP, ADSM_BAS, LDSM_PED, CDSM_CLN, W0, A0)

#### Note on walls: zero walls suffice for pure shadow; real walls are only needed for Tmrt/UTCI
- **only_shadow mode (default in this notebook)**: real walls vs all-zero walls have been verified to give **bit-for-bit identical 24-band** Shadow/Category outputs (walls only affect the irradiance received by wall surfaces, not ground shadow propagation) -> use the WALLS0/ASPECT0 generated by prep directly and save ~7-8 hours citywide;
- **Future Tmrt/UTCI (full physics)**: wall temperature / reflected radiation need real walls -- recompute them from DSMremain with the 12-core parallel script in the next cell (mandatory for a small single-tile image; for a large multi-tile image you may also omit the wall parameters and let the package compute them per tile).

In [ ]:
# ===== (only needed for Tmrt/UTCI) recompute real walls on 12 cores in parallel, bit-for-bit equivalent to the in-package algorithm =====
# import subprocess, sys
# script = r'D:\Claude\SVI_FFW\output\step3_adsm\step3b_walls_parallel.py'
# WH_OUT = name('WHremain'); WA_OUT = name('WAremain')
# r = subprocess.run([sys.executable, script,
#                     os.path.join(raster_path, DSM_REM),
#                     os.path.join(raster_path, WH_OUT),
#                     os.path.join(raster_path, WA_OUT), '12'],
#                    capture_output=True, text=True)
# print(r.stdout[-1500:]); print(r.stderr[-500:])

#### Shadow Category [B+T+S+**A**] -- single run

In [ ]:
from solweig_gpu import thermal_comfort
import time

start = time.perf_counter()
thermal_comfort(
    base_path = raster_path,
    selected_date_str = date,

    # ===== Final configuration identical to the successful citywide run (98.9 min / 77 tiles) =====
    building_dsm_filename = DSM_REM,        # ★ DEM + building_remain.height
    dem_filename   = DEM_IN,
    trees_filename = CDSM_CLN,              # ★ cleaned canopy (height above ground)
    landcover_filename = None,              # not needed for only_shadow; switch to LC_IN for Tmrt/UTCI
    ldsm_filename  = LDSM_PED,              # ★ covered linkway pednet @3 m
    arcade_filename      = ADSM_TOP,        # ★ arcade box top
    arcade_base_filename = ADSM_BAS,        # ★ arcade box base
    building_remain_filename = BREMAIN,     # ★ class 12 mask
    wallheight_filename = W0,               # ★ zero walls (verified bit-for-bit equivalent in only_shadow mode)
    wallaspect_filename = A0,

    tile_size = 4000,
    overlap   = 100,

    use_own_met = True,
    start_time = f'{date} 00:00:00',
    end_time   = f'{date} 23:00:00',
    own_met_file = wea_path,

    save_tmrt = False,
    save_shadow = True,
    shadow_category = True,
    only_shadow = True,
    skip_sparse_tiles = True,
    reuse_tiles = True,        # reuse existing tiles; after changing the base rasters, delete the corresponding layer folders under processed_inputs
)
print(f'elapsed {(time.perf_counter()-start)/60:.1f} min')

#### Second run (reuses tiles and walls; use when rerunning with a different date/parameters)

In [ ]:
# ===== Second run (all tiles reused; use when rerunning with a different date) =====
# from solweig_gpu import thermal_comfort
# import time
# start = time.perf_counter()
# thermal_comfort(
#     base_path = raster_path, selected_date_str = date,
#     building_dsm_filename = DSM_REM, dem_filename = DEM_IN, trees_filename = CDSM_CLN,
#     landcover_filename = None, ldsm_filename = LDSM_PED,
#     arcade_filename = ADSM_TOP, arcade_base_filename = ADSM_BAS,
#     building_remain_filename = BREMAIN,
#     wallheight_filename = W0, wallaspect_filename = A0,
#     tile_size = 4000, overlap = 100,
#     use_own_met = True, start_time = f'{date} 00:00:00', end_time = f'{date} 23:00:00',
#     own_met_file = wea_path,
#     save_tmrt = False, save_shadow = True, shadow_category = True, only_shadow = True,
#     skip_sparse_tiles = True,
#     reuse_tiles = True,   # ★ reuse everything (the met files are regenerated automatically each time)
# )
# print(f'elapsed {(time.perf_counter()-start)/60:.1f} min')

#### Result check (1) -- hourly class statistics over all tiles

In [ ]:
import glob, numpy as np
from osgeo import gdal
NAMES = {0:'sunlit',1:'bldg',2:'veg',3:'b+v',4:'ldsm',5:'b+ldsm',6:'v+ldsm',7:'b+v+ldsm',
         8:'ARCADE',9:'veg+arc',10:'ldsm+arc',11:'v+l+arc',12:'FOOTPRINT'}
TILE = 4000   # same as tile_size; only the core of each tile is counted, the overlap band is clipped to avoid double counting
cats = sorted(glob.glob(os.path.join(raster_path, 'output_folder', '*', 'Category_*.tif')))
print(f'tiles: {len(cats)}')
ref = gdal.Open(os.path.join(raster_path, DEM_IN))
CITY_W, CITY_H = ref.RasterXSize, ref.RasterYSize; ref = None
tot = None
for p in cats:
    key = os.path.basename(os.path.dirname(p))
    x, y = map(int, key.split('_'))
    cw, chh = min(TILE, CITY_W - x), min(TILE, CITY_H - y)
    ds = gdal.Open(p); nb = ds.RasterCount
    if tot is None: tot = np.zeros((nb, 13), dtype=np.int64)
    for b in range(nb):
        arr = ds.GetRasterBand(b+1).ReadAsArray()[:chh, :cw]
        u, c = np.unique(arr, return_counts=True)
        for v, k in zip(u.tolist(), c.tolist()):
            if 0 <= v <= 12: tot[b, v] += k
    ds = None
print('hr    shade%  ' + ' '.join(f'{NAMES[k]:>11}' for k in range(13)))
for b in range(tot.shape[0]):
    denom = int(tot[b, 0:12].sum()); shade = int(tot[b, 1:12].sum())
    pct = shade/denom*100 if denom else 0.0
    print(f'{b:02d}  {pct:>8.2f}  ' + ' '.join(f'{tot[b,k]:>11}' for k in range(13)))
# Basis: shadow share = (classes 1-11)/(classes 0-11); the denominator excludes class 12 building footprint; sea pixels are included, a land basis needs an additional DEM>0 mask

#### Result check (2) -- single-tile daytime-hour preview (palette rendering)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
tile = cats[0]                       # change to the tile you want to inspect
HOUR = 14                            # local time (band number = HOUR+1)
ds = gdal.Open(tile); arr = ds.GetRasterBand(HOUR+1).ReadAsArray(); ds = None
cmap = ListedColormap(['#ffffcc','#9a9a9a','#74c476','#4a7f4a','#fdae6b','#b07040','#8fae5a','#6b5b3a',
                       '#d40000','#c065c0','#e08020','#7b3fa0','#5a5a5a'])
norm = BoundaryNorm(np.arange(-0.5,13.5,1), cmap.N)
fig, ax = plt.subplots(figsize=(11,11))
ax.imshow(arr, cmap=cmap, norm=norm, interpolation='nearest')
ax.set_title(f'{os.path.basename(tile)}  band {HOUR+1} ({HOUR}:00 local)')
ax.set_xticks([]); ax.set_yticks([])
fig.legend(handles=[Patch(fc=cmap(k), label=f'{k} {NAMES[k]}') for k in range(13)],
           loc='lower center', ncol=7, fontsize=9)
plt.show()
# QGIS viewing tip: pick a daytime band (9-19) + "Paletted/Unique values" rendering; band 1 = 00:00 being all 0 is normal.